In [1]:
!pip install -q datasets tokenizers

In [2]:
import math
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

Using device: cuda


---
# 1) BERT
> *"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding"*
> — Devlin, Chang, Lee & Toutanova (2019) · [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

BERT introduced the **masked language model (MLM)** pre-training objective, enabling
*bidirectional* context — unlike GPT (left-to-right) or ELMo (shallow bidirectional).

**Key innovations:**
1. **WordPiece tokenization** — handles open vocabulary via subword decomposition
2. **Bidirectional self-attention** — every token attends to every other token
3. **Two pre-training tasks** — MLM (predict masked tokens) + NSP (next sentence prediction)
4. **Fine-tuning paradigm** — pre-train once, fine-tune for downstream tasks

**Sample input / output:**
```
Input:  "The cat [MASK] on the mat"
Output: "The cat sat on the mat"   (MLM recovers masked token)
```

## 1.1 WordPiece Tokenization

$$\text{score}(a, b) = \frac{\text{freq}(a, b)}{\text{freq}(a) \times \text{freq}(b)}$$

Unlike BPE which uses raw pair frequency, WordPiece **normalizes by individual token
frequency** — this prioritizes merging rare-but-consistently-co-occurring pairs.

**Sample input → output:**
```
"unhappiness" → ["un", "##hap", "##pi", "##ness"]
```
The `##` prefix indicates a continuation subword (not a word start).

**Algorithm steps:**
1. Pre-tokenize text into words (whitespace + punctuation split)
2. Split each word into characters; prefix non-initial chars with `##`
3. Compute pair scores across all word splits
4. Merge the highest-scoring pair; add merged token to vocabulary
5. Repeat until `vocab_size` is reached

In [20]:
from transformers import AutoTokenizer

# Use a pre-trained tokenizer just for pre-tokenization (word splitting)
pretokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

# Count word frequencies across the corpus
# word_freqs: {word_str -> int}
word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = pretokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    for word, _offset in words_with_offsets:
        word_freqs[word] += 1

print("Word frequencies:", dict(word_freqs))

Word frequencies: {'This': 3, 'is': 2, 'the': 1, 'Hugging': 1, 'Face': 1, 'Course': 1, '.': 4, 'chapter': 1, 'about': 1, 'tokenization': 1, 'section': 1, 'shows': 1, 'several': 1, 'tokenizer': 1, 'algorithms': 1, 'Hopefully': 1, ',': 1, 'you': 1, 'will': 1, 'be': 1, 'able': 1, 'to': 1, 'understand': 1, 'how': 1, 'they': 1, 'are': 1, 'trained': 1, 'and': 1, 'generate': 1, 'tokens': 1}


In [21]:
# Build character-level alphabet: first char as-is, continuation chars with ## prefix
# alphabet: list[str]
alphabet = []
for word in word_freqs.keys():
    if word[0] not in alphabet:
        alphabet.append(word[0])
    for letter in word[1:]:
        if f"##{letter}" not in alphabet:
            alphabet.append(f"##{letter}")
alphabet.sort()

# Initial vocab: special tokens + character alphabet
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

# Initialize character-level splits for each word
# splits: {word_str -> list[token_str]}
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}
print(f"\nInitial splits for 'about': {splits.get('about', 'N/A')}")


Initial splits for 'about': ['a', '##b', '##o', '##u', '##t']


In [22]:
def compute_pair_scores(splits):
    """Compute WordPiece pair scores: score(a,b) = freq(a,b) / (freq(a) * freq(b))."""
    # letter_freqs: {token -> int}, pair_freqs: {(tok_a, tok_b) -> int}
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            letter_freqs[split[0]] += freq
            continue
        for i in range(len(split) - 1):
            pair_freqs[(split[i], split[i + 1])] += freq
            letter_freqs[split[i]] += freq
        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }
    return scores


def merge_pair(a, b, splits):
    """Merge token pair (a, b) in all word splits."""
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merged = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merged] + split[i + 2:]
            else:
                i += 1
        splits[word] = split
    return splits


# Iteratively merge best pairs until target vocab_size
# vocab_size: int — target vocabulary size
target_vocab_size = 70
while len(vocab) < target_vocab_size:
    scores = compute_pair_scores(splits)
    best_pair, max_score = max(scores.items(), key=lambda x: x[1])
    splits = merge_pair(*best_pair, splits)
    new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )
    vocab.append(new_token)

print(f"\nFinal vocab ({len(vocab)} tokens): {vocab}")


Final vocab (70 tokens): ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##a', '##b', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##v', '##w', '##y', '##z', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'g', 'h', 'i', 's', 't', 'u', 'w', 'y', 'ab', '##fu', 'Fa', 'Fac', '##ct', '##ful', '##full', '##fully', 'Th', 'ch', '##hm', 'cha', 'chap', 'chapt', '##thm', 'Hu', 'Hug', 'Hugg', 'sh', 'th', 'is', '##thms', '##za', '##zat', '##ut']


In [23]:
def encode_word(word):
    """Greedy longest-prefix-match encoding using our WordPiece vocab."""
    tokens = []
    while len(word) > 0:
        i = len(word)
        while i > 0 and word[:i] not in vocab:
            i -= 1
        if i == 0:
            return ["[UNK]"]
        tokens.append(word[:i])
        word = word[i:]
        if len(word) > 0:
            word = f"##{word}"
    return tokens


print(f"\n'Hugging' -> {encode_word('Hugging')}")
print(f"'HOgging' -> {encode_word('HOgging')}")


'Hugging' -> ['Hugg', '##i', '##n', '##g']
'HOgging' -> ['[UNK]']


## 1.2 BERT Embedding Layer

BERT's input representation is the **element-wise sum** of three embeddings:

| Embedding | Purpose | Shape |
|-----------|---------|-------|
| Token | Map token IDs to dense vectors | `(batch_num, seq_len)` → `(batch_num, seq_len, embed_dim)` |
| Position | Encode absolute position (sinusoidal or learned) | `(1, max_len, embed_dim)` broadcast to batch |
| Segment | Distinguish sentence A from sentence B | `(batch_num, seq_len)` → `(batch_num, seq_len, embed_dim)` |

The original BERT uses **learned** positional embeddings, but here we implement the
**sinusoidal** variant from Vaswani et al. (2017) for educational clarity:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

**Sample input → output:**
```
Token IDs:  [101, 2023, 2003, 102]     → (batch_num, seq_len=4)
Segment:    [1,   1,    1,    1]        → (batch_num, seq_len=4)
Output:     (batch_num, seq_len=4, embed_dim=768) — dense representation
```

In [4]:
class PositionalEmbedding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, embed_dim, max_len=512):
        super().__init__()
        # pe: (max_len, embed_dim)
        pe = torch.zeros(max_len, embed_dim)

        # position: (max_len, 1), div_term: (embed_dim // 2,)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2, dtype=torch.float)
            * (-math.log(10000.0) / embed_dim)
        )

        # Apply sin to even indices, cos to odd indices
        # (max_len, embed_dim // 2)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # (1, max_len, embed_dim) — register as buffer (not a parameter)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x: (batch_num, seq_len, embed_dim) — used only for seq_len
        # (1, max_len, embed_dim) → slice to (1, seq_len, embed_dim)
        return self.pe[:, : x.size(1), :]


class BERTEmbedding(nn.Module):
    """Sum of token + positional + segment embeddings with dropout."""

    def __init__(self, vocab_size, embed_dim, max_len=512, dropout=0.1):
        super().__init__()
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        self.token = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Segment: 0=pad, 1=sentence_A, 2=sentence_B
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        self.segment = nn.Embedding(3, embed_dim, padding_idx=0)

        # (1, max_len, embed_dim) — broadcast across batch
        self.position = PositionalEmbedding(embed_dim, max_len)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, token_ids, segment_ids):
        # token_ids:   (batch_num, seq_len)
        # segment_ids: (batch_num, seq_len)

        # (batch_num, seq_len, embed_dim)
        tok_emb = self.token(token_ids)

        # (1, seq_len, embed_dim) — broadcasts to (batch_num, seq_len, embed_dim)
        pos_emb = self.position(tok_emb)

        # (batch_num, seq_len, embed_dim)
        seg_emb = self.segment(segment_ids)

        # Element-wise sum → (batch_num, seq_len, embed_dim)
        x = tok_emb + pos_emb + seg_emb
        # LayerNorm + dropout → (batch_num, seq_len, embed_dim)
        return self.dropout(self.norm(x))


# Smoke test
_emb = BERTEmbedding(vocab_size=30000, embed_dim=768, max_len=128)
_tok = torch.randint(1, 30000, (2, 32))
_seg = torch.ones(2, 32, dtype=torch.long)
print(f"BERTEmbedding output: {_emb(_tok, _seg).shape}")  # (2, 32, 768)

BERTEmbedding output: torch.Size([2, 32, 768])


## 1.3 Multi-Head Self-Attention

Each attention head computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Multiple heads run in parallel, each with its own learned projections, then their
outputs are concatenated and projected:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

**Sample shapes for `embed_dim=768`, `num_heads=12`:**
```
Input:  (batch_num, seq_len, 768)
Split:  (batch_num, 12, seq_len, 64)    # 12 heads × 64 dim each
Scores: (batch_num, 12, seq_len, seq_len)
Output: (batch_num, seq_len, 768)        # heads concatenated + projected
```

In [5]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention with padding mask support."""

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        # head_dim = embed_dim // num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = math.sqrt(self.head_dim)

        # Q, K, V projections: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)

        # Output projection: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_num, seq_len, embed_dim)
        # mask: (batch_num, 1, 1, seq_len) — 1 for valid, 0 for padding
        batch_num, seq_len, _ = x.shape

        # Linear projections: (batch_num, seq_len, embed_dim)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape to (batch_num, num_heads, seq_len, head_dim)
        Q = Q.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, seq_len)
        # → scores: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Mask padding positions with -inf before softmax
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # Softmax → attention weights: (batch_num, num_heads, seq_len, seq_len)
        attn_weights = self.dropout(F.softmax(scores, dim=-1))

        # Weighted sum of values
        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → context: (batch_num, num_heads, seq_len, head_dim)
        context = torch.matmul(attn_weights, V)

        # Concatenate heads
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, embed_dim)
        context = context.transpose(1, 2).contiguous().view(batch_num, seq_len, -1)

        # Output projection: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        return self.W_o(context)

## 1.4 Feed-Forward Network & Encoder Block

Each encoder layer has two sub-layers:
1. **Multi-Head Self-Attention** with residual connection + LayerNorm
2. **Position-wise FFN** with residual connection + LayerNorm

The FFN applies two linear transformations with GELU activation:

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

BERT uses **Post-LayerNorm** (norm after residual addition):
$$\text{output} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

**Note:** This contrasts with the **Pre-LayerNorm** used in ModernBERT (Part 3).

In [6]:
class PositionwiseFFN(nn.Module):
    """Position-wise feed-forward: expand then project back, with GELU."""

    def __init__(self, embed_dim, hidden_dim, dropout=0.1):
        super().__init__()
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch_num, seq_len, embed_dim)
        # → (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        return self.fc2(self.dropout(self.gelu(self.fc1(x))))


class BERTEncoderBlock(nn.Module):
    """Single BERT encoder block: MHA + FFN with Post-LayerNorm residuals."""

    def __init__(self, embed_dim=768, num_heads=12, hidden_dim=3072, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ffn = PositionwiseFFN(embed_dim, hidden_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_num, seq_len, embed_dim)

        # Sub-layer 1: Multi-Head Attention + residual + Post-LayerNorm
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        attn_out = self.dropout(self.attention(x, mask))
        x = self.norm1(x + attn_out)

        # Sub-layer 2: FFN + residual + Post-LayerNorm
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        ffn_out = self.dropout(self.ffn(x))
        x = self.norm2(x + ffn_out)

        # (batch_num, seq_len, embed_dim)
        return x


# Smoke test
_blk = BERTEncoderBlock()
_x = torch.randn(2, 32, 768)
_mask = torch.ones(2, 1, 1, 32)
print(f"EncoderBlock output: {_blk(_x, _mask).shape}")  # (2, 32, 768)

EncoderBlock output: torch.Size([2, 32, 768])


## 1.5 Full BERT Model with Pre-training Heads

The BERT model consists of:
1. **Embedding layer** — token + position + segment
2. **Stacked encoder blocks** — `num_layers` Transformer layers
3. **Two pre-training heads:**
   - **MLM head**: predicts masked tokens → `(batch_num, seq_len, vocab_size)`
   - **NSP head**: predicts if sentence B follows A → `(batch_num, 2)`

The MLM head operates on every position, while the NSP head uses only the `[CLS]` token
representation (position 0).

**Sample flow:**
```
[CLS] The cat [MASK] on [SEP] The mat [SEP]
  ↓      ↓    ↓    ↓    ↓     ↓    ↓    ↓
 NSP    ...  ...  "sat" ...   ...  ...  ...    ← MLM predicts at [MASK] positions
  ↓
is_next=1
```

In [7]:
class BERT(nn.Module):
    """BERT encoder backbone: embeddings + stacked Transformer layers."""

    def __init__(
        self,
        vocab_size,
        embed_dim=768,
        num_layers=6,
        num_heads=12,
        hidden_dim=3072,
        max_len=512,
        dropout=0.1,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = BERTEmbedding(vocab_size, embed_dim, max_len, dropout)
        self.layers = nn.ModuleList(
            [
                BERTEncoderBlock(embed_dim, num_heads, hidden_dim, dropout)
                for _ in range(num_layers)
            ]
        )

    def forward(self, token_ids, segment_ids):
        # token_ids:   (batch_num, seq_len)
        # segment_ids: (batch_num, seq_len)

        # Padding mask: (batch_num, 1, 1, seq_len) — broadcasts across heads and query positions
        mask = (token_ids != 0).unsqueeze(1).unsqueeze(2)

        # Embeddings: (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        x = self.embedding(token_ids, segment_ids)

        # Pass through each encoder block sequentially
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        for layer in self.layers:
            x = layer(x, mask)

        return x


class MaskedLanguageModel(nn.Module):
    """MLM head: predict original token at each position."""

    def __init__(self, embed_dim, vocab_size):
        super().__init__()
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        # x: (batch_num, seq_len, embed_dim)
        # → (batch_num, seq_len, vocab_size)
        return self.linear(x)


class NextSentencePrediction(nn.Module):
    """NSP head: binary classification using [CLS] token."""

    def __init__(self, embed_dim):
        super().__init__()
        # (batch_num, embed_dim) → (batch_num, 2)
        self.linear = nn.Linear(embed_dim, 2)

    def forward(self, x):
        # Use [CLS] token at position 0
        # x: (batch_num, seq_len, embed_dim) → x[:, 0]: (batch_num, embed_dim)
        # → (batch_num, 2)
        return self.linear(x[:, 0])


class BERTLM(nn.Module):
    """BERT with MLM + NSP pre-training heads."""

    def __init__(self, bert, vocab_size):
        super().__init__()
        self.bert = bert
        self.mlm = MaskedLanguageModel(bert.embed_dim, vocab_size)
        self.nsp = NextSentencePrediction(bert.embed_dim)

    def forward(self, token_ids, segment_ids):
        # token_ids:   (batch_num, seq_len)
        # segment_ids: (batch_num, seq_len)

        # Encoder output: (batch_num, seq_len, embed_dim)
        h = self.bert(token_ids, segment_ids)

        # MLM logits: (batch_num, seq_len, vocab_size)
        mlm_logits = self.mlm(h)
        # NSP logits: (batch_num, 2)
        nsp_logits = self.nsp(h)

        return mlm_logits, nsp_logits


# Smoke test
VOCAB_SIZE = 30000
bert = BERT(VOCAB_SIZE, embed_dim=256, num_layers=2, num_heads=8, hidden_dim=512, max_len=64)
bert_lm = BERTLM(bert, VOCAB_SIZE)

_tok = torch.randint(1, VOCAB_SIZE, (4, 32))
_seg = torch.ones(4, 32, dtype=torch.long)
_mlm, _nsp = bert_lm(_tok, _seg)
print(f"MLM logits: {_mlm.shape}")  # (4, 32, 30000)
print(f"NSP logits: {_nsp.shape}")  # (4, 2)
print(f"Total parameters: {sum(p.numel() for p in bert_lm.parameters()):,}")

MLM logits: torch.Size([4, 32, 30000])
NSP logits: torch.Size([4, 2])
Total parameters: 16,446,002


## 1.6 Pre-training: Dataset & Training Loop

**MLM masking strategy** (from the original paper):
- 15% of tokens are selected for prediction
- Of those: 80% replaced with `[MASK]`, 10% replaced with random token, 10% kept unchanged

**NSP task:**
- 50% of sentence pairs are actual consecutive sentences (label=1)
- 50% are random pairs (label=0)

**Training uses a warmup learning rate schedule:**

$$\text{lr}(t) = d_{\text{model}}^{-0.5} \cdot \min(t^{-0.5},\; t \cdot t_{\text{warmup}}^{-1.5})$$

We demonstrate on synthetic data to keep the notebook self-contained.

In [8]:
class BERTPretrainDataset(Dataset):
    """Synthetic pre-training dataset with MLM + NSP tasks."""

    def __init__(self, num_samples=1000, vocab_size=30000, seq_len=32):
        self.num_samples = num_samples
        self.vocab_size = vocab_size
        self.seq_len = seq_len

        # Generate random sentence pairs: list of (sent_a_ids, sent_b_ids)
        self.pairs = []
        for _ in range(num_samples):
            len_a = random.randint(5, seq_len // 2 - 2)
            len_b = random.randint(5, seq_len // 2 - 2)
            sent_a = [random.randint(5, vocab_size - 1) for _ in range(len_a)]
            sent_b = [random.randint(5, vocab_size - 1) for _ in range(len_b)]
            self.pairs.append((sent_a, sent_b))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        CLS, SEP, PAD, MASK = 1, 2, 0, 3

        sent_a, sent_b = self.pairs[idx]

        # 50% chance: use actual pair (is_next=1), else random pair (is_next=0)
        if random.random() < 0.5:
            is_next = 1
        else:
            rand_idx = random.randint(0, len(self.pairs) - 1)
            sent_b = self.pairs[rand_idx][1]
            is_next = 0

        # Build input: [CLS] sent_a [SEP] sent_b [SEP]
        tokens = [CLS] + sent_a + [SEP] + sent_b + [SEP]
        segments = [1] * (len(sent_a) + 2) + [2] * (len(sent_b) + 1)

        # MLM masking: 15% of non-special tokens
        mlm_labels = [PAD] * len(tokens)
        for i in range(len(tokens)):
            if tokens[i] in (CLS, SEP, PAD):
                continue
            if random.random() < 0.15:
                mlm_labels[i] = tokens[i]
                r = random.random()
                if r < 0.8:
                    tokens[i] = MASK
                elif r < 0.9:
                    tokens[i] = random.randint(5, self.vocab_size - 1)

        # Truncate and pad to seq_len
        tokens = tokens[: self.seq_len]
        segments = segments[: self.seq_len]
        mlm_labels = mlm_labels[: self.seq_len]

        pad_len = self.seq_len - len(tokens)
        tokens += [PAD] * pad_len
        segments += [PAD] * pad_len
        mlm_labels += [PAD] * pad_len

        return {
            "input_ids": torch.tensor(tokens, dtype=torch.long),
            "segment_ids": torch.tensor(segments, dtype=torch.long),
            "mlm_labels": torch.tensor(mlm_labels, dtype=torch.long),
            "is_next": torch.tensor(is_next, dtype=torch.long),
        }


# Create dataset and dataloader
train_dataset = BERTPretrainDataset(num_samples=2000, vocab_size=VOCAB_SIZE, seq_len=32)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
sample = next(iter(train_loader))
print(f"input_ids:   {sample['input_ids'].shape}")    # (64, 32)
print(f"segment_ids: {sample['segment_ids'].shape}")   # (64, 32)
print(f"mlm_labels:  {sample['mlm_labels'].shape}")    # (64, 32)
print(f"is_next:     {sample['is_next'].shape}")       # (64,)

input_ids:   torch.Size([64, 32])
segment_ids: torch.Size([64, 32])
mlm_labels:  torch.Size([64, 32])
is_next:     torch.Size([64])


In [9]:
class BERTTrainer:
    """BERT pre-training loop with warmup learning rate schedule."""

    def __init__(self, model, train_loader, lr=1e-4, warmup_steps=100, device="cpu"):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader

        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        self.mlm_criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.nsp_criterion = nn.CrossEntropyLoss()

        self.warmup_steps = warmup_steps
        self.step_count = 0
        self.base_lr = lr

    def _update_lr(self):
        self.step_count += 1
        if self.step_count < self.warmup_steps:
            lr = self.base_lr * self.step_count / self.warmup_steps
        else:
            lr = self.base_lr * (self.warmup_steps / self.step_count) ** 0.5
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0

        for i, batch in enumerate(self.train_loader):
            batch = {k: v.to(self.device) for k, v in batch.items()}

            # Forward pass
            # mlm_logits: (batch_num, seq_len, vocab_size)
            # nsp_logits: (batch_num, 2)
            mlm_logits, nsp_logits = self.model(batch["input_ids"], batch["segment_ids"])

            # MLM loss: (batch_num, vocab_size, seq_len) vs (batch_num, seq_len)
            mlm_loss = self.mlm_criterion(
                mlm_logits.transpose(1, 2), batch["mlm_labels"]
            )
            # NSP loss: (batch_num, 2) vs (batch_num,)
            nsp_loss = self.nsp_criterion(nsp_logits, batch["is_next"])
            loss = mlm_loss + nsp_loss

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            self._update_lr()

            total_loss += loss.item()
            if (i + 1) % 10 == 0:
                print(
                    f"  Epoch {epoch+1} | Step {i+1}/{len(self.train_loader)} | "
                    f"Loss: {total_loss / (i+1):.4f}"
                )

        avg_loss = total_loss / len(self.train_loader)
        print(f"Epoch {epoch+1} complete — avg_loss: {avg_loss:.4f}")
        return avg_loss


# Train for 2 epochs on synthetic data
trainer = BERTTrainer(bert_lm, train_loader, lr=1e-3, warmup_steps=50, device=str(DEVICE))
for epoch in range(2):
    trainer.train_epoch(epoch)

  Epoch 1 | Step 10/32 | Loss: 12.5770
  Epoch 1 | Step 20/32 | Loss: 11.9639
  Epoch 1 | Step 30/32 | Loss: 11.7233
Epoch 1 complete — avg_loss: 11.6878
  Epoch 2 | Step 10/32 | Loss: 11.1113
  Epoch 2 | Step 20/32 | Loss: 11.1122
  Epoch 2 | Step 30/32 | Loss: 11.0863
Epoch 2 complete — avg_loss: 11.0936


---
# 2) DeBERTa

DeBERTa improves BERT with two key ideas:

### a. Disentangled Attention
Standard attention computes $(C_i + P_i)(C_j + P_j)^T$, collapsing content and position
into a single representation. DeBERTa **disentangles** them into three explicit terms:

| Term | Formula | Meaning |
|------|---------|---------|
| **c2c** | $C_i C_j^T$ | Content-to-content (standard attention) |
| **c2p** | $C_i P_j^T$ | Query content attends to key positions |
| **p2c** | $P_i C_j^T$ | Query position attends to key content |

The **p2p** term ($P_i P_j^T$) is dropped because it is input-independent (provides
no useful signal about specific inputs).

### b. Enhanced Mask Decoder (EMD)
Absolute position information is injected **only in the final decoder layer**, not
in every encoder layer. This allows the encoder to learn content-relative-position
interactions without being biased by absolute positions.

### c. DeBERTa v2/v3 (Later Improvements)
- **v2**: Replaced absolute position with **span-based positional encoding**
- **v3**: Replaced MLM with **ELECTRA-style replaced token detection (RTD)** +
  gradient-disentangled embedding sharing (GDES) for more sample-efficient pre-training

```
BERT attention:   score = (content + position) × (content + position)ᵀ
DeBERTa attention: score = c2c + c2p + p2c   (disentangled)
```

## 2.1 Log-Bucketed Relative Positions

DeBERTa uses **relative** position encoding instead of absolute. For efficiency with
long sequences, positions beyond a threshold are **log-bucketed** — nearby positions
get exact encoding while distant positions are grouped into buckets.

Given max relative distance $k$ and bucket count $b$:
- Positions in $[-b/2, b/2]$ → exact relative offset
- Positions outside this range → compressed via $\log$ bucketing

$$\text{bucket}(\delta) = \begin{cases} \delta & \text{if } |\delta| \le b/2 \\ \text{sign}(\delta) \cdot \left\lfloor b/2 + \frac{\log(|\delta|/b \cdot 2)}{\log(k/b \cdot 2)} \cdot (b/2) \right\rfloor & \text{otherwise} \end{cases}$$

This gives **finer granularity for nearby tokens** (which matter more for syntax/semantics)
and **coarser encoding for distant tokens**.

**Sample:**
```
Relative positions: [-10, -5, -2, -1, 0, 1, 2, 5, 10]
After bucketing:    [-8,  -5, -2, -1, 0, 1, 2, 5, 8]   (distant ones compressed)
```

In [10]:
def make_log_bucket_position(relative_pos, bucket_size, max_position):
    """
    Compress relative positions using log-bucketing.
    Nearby positions keep exact values; distant ones are grouped into log-spaced buckets.

    Args:
        relative_pos: (query_size, key_size) — raw relative position offsets
        bucket_size:  int — number of buckets
        max_position: int — maximum absolute relative distance

    Returns:
        (query_size, key_size) — bucketed relative position indices
    """
    # sign: (query_size, key_size) — -1, 0, or 1
    sign = torch.sign(relative_pos)
    mid = bucket_size // 2

    # absolute relative distances: (query_size, key_size)
    abs_pos = torch.where(
        (relative_pos < mid) & (relative_pos > -mid),
        torch.tensor(mid - 1).float(),
        torch.abs(relative_pos).float(),
    )

    # Log-scale mapping for positions beyond mid
    # Maps [mid, max_position] → [mid, bucket_size-1] logarithmically
    log_pos = (
        torch.ceil(
            torch.log(abs_pos / mid) / math.log(max_position / mid) * (mid - 1)
        )
        + mid
    )

    # Clamp to valid bucket range
    bucket_pos = torch.where(
        abs_pos <= mid, relative_pos.float(), log_pos * sign
    ).long()

    return bucket_pos


def build_relative_position(
        query_size, key_size, bucket_size=-1, max_position=-1
    ):
    """
    Build the relative position matrix for Q×K attention.

    Returns:
        (1, query_size, key_size) — relative position indices
    """

    # q_ids: (query_size,), k_ids: (key_size,)
    q_ids = torch.arange(query_size)
    k_ids = torch.arange(key_size)

    # Outer subtraction → (query_size, key_size)
    rel_pos = q_ids[:, None] - k_ids[None, :]

    # Apply log-bucketing if configured
    if bucket_size > 0 and max_position > 0:
        rel_pos = make_log_bucket_position(rel_pos, bucket_size, max_position)

    # (1, query_size, key_size) — add batch dim for broadcasting
    return rel_pos.unsqueeze(0)


# Demo: show raw vs bucketed positions for seq_len=16
raw_pos = build_relative_position(16, 16)
bucketed_pos = build_relative_position(16, 16, bucket_size=8, max_position=64)
print("Raw relative positions (row 0):", raw_pos[0, 0].tolist())
print("Bucketed positions    (row 0):", bucketed_pos[0, 0].tolist())

Raw relative positions (row 0): [0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15]
Bucketed positions    (row 0): [0, -1, -2, -3, -4, -5, -5, -5, -5, -5, -5, -6, -6, -6, -6, -6]


## 2.2 Disentangled Self-Attention

![](https://towardsdatascience.com/wp-content/uploads/2023/11/1xt8ceuK0eC62VRYF05q8tw.png)

The core innovation: separate projections for **content** and **position**, computing
three attention score matrices that are summed:

$$A_{i,j} = \underbrace{H_i W_q^c (H_j W_k^c)^T}_{\text{c2c: content×content}} + \underbrace{H_i W_q^c (P_{i|j} W_k^p)^T}_{\text{c2p: content×position}} + \underbrace{P_{j|i} W_q^p (H_j W_k^c)^T}_{\text{p2c: position×content}}$$

Where:
- $H_i$ = content (hidden state) at position $i$
- $P_{i|j}$ = relative position embedding for the offset $i - j$
- $W_q^c, W_k^c$ = content query/key projections
- $W_q^p, W_k^p$ = position query/key projections

**Implementation detail**: The c2p and p2c terms use `torch.gather` to efficiently
index into the score matrix at the correct relative positions, avoiding explicit
loops over all $(i, j)$ pairs.

In [11]:
class DisentangledSelfAttention(nn.Module):
    """
    DeBERTa's disentangled self-attention with c2c + c2p + p2c scoring.
    Reference: He et al., 2021 — Section 3.
    """

    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        max_relative_positions=512,
        position_buckets=256,
        dropout=0.1,
    ):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_relative_positions = max_relative_positions
        self.position_buckets = position_buckets

        # Content Q, K, V projections
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.W_q_c = nn.Linear(embed_dim, embed_dim)
        self.W_k_c = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)

        # Position Q, K projections (for disentangled terms)
        # (1, pos_ebd_size, embed_dim) → (1, pos_ebd_size, embed_dim)
        self.W_q_p = nn.Linear(embed_dim, embed_dim)
        self.W_k_p = nn.Linear(embed_dim, embed_dim)

        # Relative positional embedding table
        pos_ebd_size = position_buckets * 2
        # (pos_ebd_size, embed_dim)
        self.rel_embeddings = nn.Embedding(pos_ebd_size, embed_dim)

        self.output_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # scale_factor = sqrt(3 * head_dim) for 3 attention terms
        self.scale = math.sqrt(3.0 * self.head_dim)

    def _split_heads(self, x):
        """(batch_num, seq_len, embed_dim) → (batch_num, num_heads, seq_len, head_dim)"""
        b, s, _ = x.shape
        return x.view(b, s, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, hidden_states, mask=None):
        # hidden_states: (batch_num, seq_len, embed_dim)
        # mask: (batch_num, 1, 1, seq_len)
        batch_num, seq_len, _ = hidden_states.shape

        # Content projections
        # Q_c, K_c, V: (batch_num, num_heads, seq_len, head_dim)
        Q_c = self._split_heads(self.W_q_c(hidden_states))
        K_c = self._split_heads(self.W_k_c(hidden_states))
        V = self._split_heads(self.W_v(hidden_states))

        # Build relative position indices: (1, seq_len, seq_len)
        rel_pos = build_relative_position(
            seq_len,
            seq_len,
            bucket_size=self.position_buckets,
            max_position=self.max_relative_positions,
        ).to(hidden_states.device)

        # Shift to non-negative indices for embedding lookup
        pos_ebd_size = self.position_buckets * 2
        # (1, seq_len, seq_len) — indices into [0, pos_ebd_size)
        rel_pos_idx = torch.clamp(rel_pos + pos_ebd_size // 2, 0, pos_ebd_size - 1)

        # Look up relative position embeddings
        # (1, seq_len, seq_len) → (1, seq_len, seq_len, embed_dim) via embedding table
        # We use a flattened gather approach for efficiency
        rel_emb = self.rel_embeddings.weight  # (pos_ebd_size, embed_dim)

        # ─── Term 1: Content-to-Content (c2c) ───
        # → c2c: (batch_num, num_heads, seq_len, seq_len)
        c2c = torch.matmul(Q_c, K_c.transpose(-2, -1))

        # ─── Term 2: Content-to-Position (c2p) ───
        # Project all position embeddings through K_p
        # (pos_ebd_size, embed_dim) → (1, pos_ebd_size, embed_dim)
        pos_key = self.W_k_p(rel_emb.unsqueeze(0))
        # (1, num_heads, pos_ebd_size, head_dim)
        pos_key = self._split_heads(pos_key)

        # Q_c attends to all position keys
        # (batch_num, num_heads, seq_len, head_dim) @ (1, num_heads, head_dim, pos_ebd_size)
        # → c2p_raw: (batch_num, num_heads, seq_len, pos_ebd_size)
        c2p_raw = torch.matmul(Q_c, pos_key.transpose(-2, -1))

        # Gather the correct relative position scores
        # rel_pos_idx: (1, seq_len, seq_len)
        # expand to (batch_num, num_heads, seq_len, seq_len)
        c2p_idx = rel_pos_idx.unsqueeze(1).expand(
            batch_num, self.num_heads, seq_len, seq_len
        )
        # c2p: (batch_num, num_heads, seq_len, seq_len)
        c2p = torch.gather(c2p_raw, dim=-1, index=c2p_idx)

        # ─── Term 3: Position-to-Content (p2c) ───
        # Project position embeddings through Q_p
        # (1, pos_ebd_size, embed_dim) → (1, num_heads, pos_ebd_size, head_dim)
        pos_query = self._split_heads(self.W_q_p(rel_emb.unsqueeze(0)))

        # K_c attends to all position queries
        # (batch_num, num_heads, seq_len, head_dim) @ (1, num_heads, head_dim, pos_ebd_size)
        # p2c_raw: (batch_num, num_heads, seq_len, pos_ebd_size)
        p2c_raw = torch.matmul(K_c, pos_query.transpose(-2, -1))

        # For p2c, we use negated positions (key's perspective of query's position)
        p2c_idx = torch.clamp(-rel_pos + pos_ebd_size // 2, 0, pos_ebd_size - 1)
        p2c_idx = p2c_idx.unsqueeze(1).expand(batch_num, self.num_heads, seq_len, seq_len)
        # (batch_num, num_heads, seq_len, seq_len)
        p2c = torch.gather(p2c_raw, dim=-1, index=p2c_idx).transpose(-2, -1)

        # ─── Combine all three terms ───
        # (batch_num, num_heads, seq_len, seq_len)
        scores = (c2c + c2p + p2c) / self.scale

        # Apply padding mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # Softmax → attention weights: (batch_num, num_heads, seq_len, seq_len)
        attn_weights = self.dropout(F.softmax(scores, dim=-1))

        # Weighted sum of values
        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → context: (batch_num, num_heads, seq_len, head_dim)
        context = torch.matmul(attn_weights, V)

        # Concatenate heads: (batch_num, seq_len, embed_dim)
        context = context.transpose(1, 2).contiguous().view(batch_num, seq_len, -1)

        # Output projection: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        return self.output_proj(context)


# Smoke test
_dsa = DisentangledSelfAttention(embed_dim=256, num_heads=8, position_buckets=32)
_x = torch.randn(2, 16, 256)
_mask = torch.ones(2, 1, 1, 16)
print(f"DisentangledSelfAttention output: {_dsa(_x, _mask).shape}")  # (2, 16, 256)

DisentangledSelfAttention output: torch.Size([2, 16, 256])


## 2.3 Enhanced Mask Decoder (EMD)

A key insight from DeBERTa: **absolute position is needed for token prediction, but
should NOT be injected into every encoder layer.**

*Why?* The encoder benefits from learning content-relative-position interactions
without absolute position bias. But the final MLM prediction needs to know *where*
in the sentence a token is (e.g., a word at position 0 is likely a subject).

**Solution:** Add one or more additional decoder layers that combine:
1. The encoder's output (content + relative position info)
2. Absolute positional embeddings

This is the **Enhanced Mask Decoder** — it operates only on top of the encoder stack.

```
Encoder layers (N-1):  content + relative position → hidden states
EMD layer      (1):    hidden states + absolute position → final representation
MLM head:              final representation → token predictions
```

In [12]:
class EnhancedMaskDecoder(nn.Module):
    """
    DeBERTa's EMD: injects absolute position information in the final layer only.
    Uses standard self-attention (not disentangled) since it already has rich relative
    position info from the encoder.
    Reference: He et al., 2021 — Section 4.
    """

    def __init__(self, embed_dim=768, num_heads=12, hidden_dim=3072, max_len=512, dropout=0.1):
        super().__init__()
        # Learned absolute positional embeddings
        # (max_len, embed_dim)
        self.abs_pos_embedding = nn.Embedding(max_len, embed_dim)

        # Self-attention layer that combines content + absolute position
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ffn = PositionwiseFFN(embed_dim, hidden_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm_pos = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden_states, mask=None):
        # hidden_states: (batch_num, seq_len, embed_dim) — encoder output
        batch_num, seq_len, _ = hidden_states.shape

        # Generate absolute position embeddings
        # positions: (seq_len,) → abs_emb: (1, seq_len, embed_dim) → broadcast
        positions = torch.arange(seq_len, device=hidden_states.device)
        # (1, seq_len, embed_dim)
        abs_emb = self.norm_pos(self.abs_pos_embedding(positions).unsqueeze(0))

        # Combine encoder output with absolute position via addition
        # (batch_num, seq_len, embed_dim) + (1, seq_len, embed_dim)
        # → (batch_num, seq_len, embed_dim)
        x = hidden_states + abs_emb

        # Standard self-attention + residual + LayerNorm
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        attn_out = self.dropout(self.attention(x, mask))
        x = self.norm1(x + attn_out)

        # FFN + residual + LayerNorm
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        ffn_out = self.dropout(self.ffn(x))
        x = self.norm2(x + ffn_out)
        return x

# Smoke test
_emd = EnhancedMaskDecoder(embed_dim=256, num_heads=8, hidden_dim=512, max_len=64)
_h = torch.randn(2, 16, 256)
print(f"EMD output: {_emd(_h).shape}")  # (2, 16, 256)

EMD output: torch.Size([2, 16, 256])


## 2.4 Full DeBERTa Model

Assembling all components:
1. **Token embeddings** (no positional embeddings in the encoder!)
2. **N disentangled attention encoder layers** with relative position
3. **Enhanced Mask Decoder** that injects absolute position
4. **MLM head** for pre-training

```
Input token IDs
     ↓
Token Embedding (NO position added)
     ↓
Disentangled Encoder × N (uses relative position embeddings)
     ↓
Enhanced Mask Decoder (adds absolute position)
     ↓
MLM / Classification Head
```

In [13]:
class DeBERTaEncoderBlock(nn.Module):
    """DeBERTa encoder block: disentangled attention + FFN + Post-LayerNorm."""

    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        hidden_dim=3072,
        max_relative_positions=512,
        position_buckets=256,
        dropout=0.1,
    ):
        super().__init__()
        self.attention = DisentangledSelfAttention(
            embed_dim, num_heads, max_relative_positions, position_buckets, dropout
        )
        self.ffn = PositionwiseFFN(embed_dim, hidden_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_num, seq_len, embed_dim)
        attn_out = self.dropout(self.attention(x, mask))
        x = self.norm1(x + attn_out)
        ffn_out = self.dropout(self.ffn(x))
        x = self.norm2(x + ffn_out)
        return x


class DeBERTa(nn.Module):
    """
    Full DeBERTa model: token embeddings + disentangled encoder + EMD.
    No positional embeddings in the encoder — only relative positions in attention
    and absolute positions in the EMD.
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=768,
        num_layers=6,
        num_heads=12,
        hidden_dim=3072,
        max_len=512,
        max_relative_positions=512,
        position_buckets=256,
        dropout=0.1,
    ):
        super().__init__()
        self.embed_dim = embed_dim

        # Token embedding only — no positional embedding in the encoder
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding_norm = nn.LayerNorm(embed_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        # Disentangled encoder layers (use relative position internally)
        self.encoder_layers = nn.ModuleList(
            [
                DeBERTaEncoderBlock(
                    embed_dim,
                    num_heads,
                    hidden_dim,
                    max_relative_positions,
                    position_buckets,
                    dropout,
                )
                for _ in range(num_layers)
            ]
        )

        # Enhanced Mask Decoder (injects absolute position)
        self.emd = EnhancedMaskDecoder(embed_dim, num_heads, hidden_dim, max_len, dropout)

    def forward(self, token_ids, mask=None):
        # token_ids: (batch_num, seq_len)

        # Padding mask: (batch_num, 1, 1, seq_len)
        if mask is None:
            mask = (token_ids != 0).unsqueeze(1).unsqueeze(2)

        # Token embeddings: (batch_num, seq_len, embed_dim)
        x = self.embedding_dropout(self.embedding_norm(self.token_embedding(token_ids)))

        # Disentangled encoder: each block uses relative positional attention
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        for layer in self.encoder_layers:
            x = layer(x, mask)

        # Enhanced Mask Decoder: inject absolute position
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = self.emd(x, mask)

        return x


# Build and test DeBERTa
deberta = DeBERTa(
    vocab_size=VOCAB_SIZE,
    embed_dim=256,
    num_layers=2,
    num_heads=8,
    hidden_dim=512,
    max_len=64,
    position_buckets=32,
)
_tok = torch.randint(1, VOCAB_SIZE, (2, 16))
_out = deberta(_tok)
print(f"DeBERTa output: {_out.shape}")  # (2, 16, 256)
print(f"DeBERTa parameters: {sum(p.numel() for p in deberta.parameters()):,}")

DeBERTa output: torch.Size([2, 16, 256])
DeBERTa parameters: 9,574,656



# 3) ModernBERT

> *"Smarter, Better, Faster, Longer: A Modern Bidirectional Encoder for Fast, Memory Efficient, and Long Context Finetuning and Inference"*

ModernBERT applies lessons from the LLM revolution (2020-2024) to the encoder architecture:

| Component | BERT (2019) | ModernBERT (2024) |
|-----------|-------------|-------------------|
| Position encoding | Absolute (learned) | **RoPE** (Su et al., 2021) |
| FFN activation | GELU | **GeGLU** (Shazeer, 2020) |
| Normalization | Post-LayerNorm | **Pre-LayerNorm** (more stable training) |
| Attention pattern | Full global | **Alternating local + global** |
| Padding handling | Padded batches | **Unpadding** (no wasted compute) |
| Context length | 512 tokens | **8,192 tokens** |
| Pre-training data | 3.3B tokens | **2T tokens** |
| Dropout | 0.1 during pretraining | **None** during pretraining |
| Optimizer | Adam | **StableAdamW** (Wortsman et al., 2024) |
| LR schedule | Warmup + linear decay | **Trapezoidal** (warmup → constant → decay) |

Key references for the components:
- **RoPE**: Su et al., *"RoFormer: Enhanced Transformer with Rotary Position Embedding"* (2021) · [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)
- **GeGLU**: Shazeer, *"GLU Variants Improve Transformer"* (2020) · [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- **Flash Attention**: Dao et al. (2022) · [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)

 ## 3.1 Rotary Position Embeddings (RoPE)


Instead of adding position information to token embeddings, RoPE encodes position
by **rotating** query and key vectors in 2D subspaces.

For a vector $x$ at position $m$, RoPE applies rotation:

$$f(x, m) = \begin{pmatrix} x_0 \\ x_1 \\ x_2 \\ x_3 \\ \vdots \end{pmatrix} \odot \begin{pmatrix} \cos(m\theta_0) \\ \cos(m\theta_0) \\ \cos(m\theta_1) \\ \cos(m\theta_1) \\ \vdots \end{pmatrix} + \begin{pmatrix} -x_1 \\ x_0 \\ -x_3 \\ x_2 \\ \vdots \end{pmatrix} \odot \begin{pmatrix} \sin(m\theta_0) \\ \sin(m\theta_0) \\ \sin(m\theta_1) \\ \sin(m\theta_1) \\ \vdots \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d}$ (same base frequencies as sinusoidal PE).

**Key property:** The dot product $f(q, m)^T f(k, n)$ depends only on the
**relative distance** $m - n$, not absolute positions. This gives RoPE a natural
relative position encoding without any additional parameters.

**Advantages over learned/sinusoidal PE:**
1. Naturally encodes relative positions
2. Better length generalization (can extrapolate beyond training length)
3. Compatible with linear attention variants
4. No additional parameters (pure geometric operation)

**Sample:**
```
Input:  Q at position 3 = [q0, q1, q2, q3]
Output: Rotated Q = rotate_2d([q0,q1], 3*θ₀) ⊕ rotate_2d([q2,q3], 3*θ₁)
```

In [14]:
class RotaryEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE) — Su et al., 2021.
    Pre-computes cos/sin tables; applied to Q and K before attention.
    """

    def __init__(self, head_dim, max_len=8192, base=10000.0):
        super().__init__()
        # Frequency for each pair of dimensions
        # theta: (head_dim // 2,) — geometric sequence
        theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("theta", theta)
        self._build_cache(max_len)

    def _build_cache(self, max_len):
        # positions: (max_len,)
        positions = torch.arange(max_len, dtype=self.theta.dtype)
        # freqs: (max_len, head_dim // 2) — outer product of positions and frequencies
        freqs = torch.outer(positions, self.theta)
        # Duplicate each frequency for pairs: (max_len, head_dim)
        freqs = torch.cat([freqs, freqs], dim=-1)
        # cos_cache, sin_cache: (max_len, head_dim)
        self.register_buffer("cos_cache", freqs.cos())
        self.register_buffer("sin_cache", freqs.sin())

    def forward(self, seq_len):
        # Return cos/sin for the requested sequence length
        # (seq_len, head_dim) each
        return self.cos_cache[:seq_len], self.sin_cache[:seq_len]


def rotate_half(x):
    """Rearrange pairs: [x0, x1, x2, x3, ...] → [-x1, x0, -x3, x2, ...]."""
    # x: (..., head_dim) → split into two halves
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    """
    Apply RoPE rotation to queries and keys.

    Args:
        q: (batch_num, num_heads, seq_len, head_dim)
        k: (batch_num, num_heads, seq_len, head_dim)
        cos: (seq_len, head_dim)
        sin: (seq_len, head_dim)

    Returns:
        q_rot, k_rot: same shapes as input
    """
    # Reshape cos/sin for broadcasting: (1, 1, seq_len, head_dim)
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    # Apply rotation: x * cos + rotate_half(x) * sin
    # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
    q_rot = q * cos + rotate_half(q) * sin
    k_rot = k * cos + rotate_half(k) * sin
    return q_rot, k_rot


# Demo: show that RoPE preserves relative-position property
rope = RotaryEmbedding(head_dim=64, max_len=128)
cos, sin = rope(seq_len=32)

q = torch.randn(1, 1, 32, 64)
k = torch.randn(1, 1, 32, 64)
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)

# Verify: dot product at positions (i, j) should only depend on i-j
# Compare (position 5, position 3) vs (position 10, position 8) — both offset=2
dot_5_3 = (q_rot[0, 0, 5] * k_rot[0, 0, 3]).sum().item()
dot_10_8 = (q_rot[0, 0, 10] * k_rot[0, 0, 8]).sum().item()
print(f"dot(q[5], k[3]) = {dot_5_3:.4f}")
print(f"dot(q[10], k[8]) = {dot_10_8:.4f}")
print(f"Same offset (2), values are similar: {abs(dot_5_3 - dot_10_8) < 1e-4}")
print(f"\nRoPE output shape: q={q_rot.shape}, k={k_rot.shape}")

dot(q[5], k[3]) = -6.0475
dot(q[10], k[8]) = 4.8725
Same offset (2), values are similar: False

RoPE output shape: q=torch.Size([1, 1, 32, 64]), k=torch.Size([1, 1, 32, 64])


## 3.2 GeGLU Activation (Gated Linear Unit)

Shazeer (2020) showed that **gated** FFN variants outperform standard FFN in Transformers.

**Standard FFN (BERT):**
$$\text{FFN}(x) = \text{GELU}(xW_1)W_2$$

**GeGLU FFN (ModernBERT):**
$$\text{GeGLU}(x) = (\text{GELU}(xW_{\text{gate}}) \odot xW_{\text{up}}) W_{\text{down}}$$

The gate $\text{GELU}(xW_{gate})$ acts as a **learned mask** that selectively filters
information before projection. This gives the model more expressive power.

**Note:** GeGLU uses 3 weight matrices instead of 2, so to match parameter count,
the intermediate dimension is typically scaled by $\frac{2}{3}$:
$\text{hidden\_dim}_{\text{GeGLU}} = \frac{2}{3} \times \text{hidden\_dim}_{\text{standard}}$

**Sample shapes for embed_dim=768, hidden_dim=2048:**
```
Input:  (batch_num, seq_len, 768)
Gate:   (batch_num, seq_len, 2048)  ← GELU applied
Up:     (batch_num, seq_len, 2048)  ← linear
Gated:  (batch_num, seq_len, 2048)  ← gate ⊙ up
Output: (batch_num, seq_len, 768)   ← project down
```

In [15]:
class GeGLU(nn.Module):
    """
    Gated Linear Unit with GELU activation — Shazeer (2020).
    Replaces standard FFN in ModernBERT for better expressiveness.
    """

    def __init__(self, embed_dim, hidden_dim, dropout=0.0):
        super().__init__()
        # Gate and up projections (can be fused into one matrix for efficiency)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        self.W_gate = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_up = nn.Linear(embed_dim, hidden_dim, bias=False)

        # Down projection
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        self.W_down = nn.Linear(hidden_dim, embed_dim, bias=False)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch_num, seq_len, embed_dim)

        # Gate: apply GELU to create learned mask
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        gate = self.gelu(self.W_gate(x))

        # Up: linear projection (no activation)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim)
        up = self.W_up(x)

        # Element-wise gating: gate ⊙ up
        # (batch_num, seq_len, hidden_dim) ⊙ (batch_num, seq_len, hidden_dim)
        # → (batch_num, seq_len, hidden_dim)
        gated = gate * up

        # Down projection
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, embed_dim)
        return self.W_down(self.dropout(gated))


# Compare standard FFN vs GeGLU parameter counts
embed_dim, hidden_dim = 768, 3072
std_ffn = PositionwiseFFN(embed_dim, hidden_dim)
geglu = GeGLU(embed_dim, hidden_dim)
print(f"Standard FFN params: {sum(p.numel() for p in std_ffn.parameters()):,}")
print(f"GeGLU params:        {sum(p.numel() for p in geglu.parameters()):,}")
print(f"(GeGLU has ~50% more params due to 3 matrices vs 2 — compensate with 2/3 hidden_dim)")

# Verify shapes
_x = torch.randn(2, 16, embed_dim)
print(f"\nGeGLU output: {geglu(_x).shape}")  # (2, 16, 768)

Standard FFN params: 4,722,432
GeGLU params:        7,077,888
(GeGLU has ~50% more params due to 3 matrices vs 2 — compensate with 2/3 hidden_dim)

GeGLU output: torch.Size([2, 16, 768])


## 3.3 Alternating Local + Global Attention

For long sequences (8192 tokens), full self-attention is $O(n^2)$ which is expensive.
ModernBERT uses an alternating pattern:

- **Most layers**: **Local (sliding window) attention** — each token attends only to
  tokens within a window of size $w$ (e.g., $w = 128$). Cost: $O(n \cdot w)$.
- **Every $k$-th layer**: **Global attention** — full self-attention over all tokens.
  Cost: $O(n^2)$ but only for a few layers.

This gives the model both:
1. **Efficiency** — most computation is local
2. **Long-range information flow** — periodic global layers propagate info across the full context

In ModernBERT-base (22 layers), global attention is used every 3rd layer.
In ModernBERT-large (28 layers), the pattern is similar.

**Sliding window attention mask:**
```
seq_len=8, window=3:
  q\k  0 1 2 3 4 5 6 7
   0  [1 1 0 0 0 0 0 0]
   1  [1 1 1 0 0 0 0 0]
   2  [0 1 1 1 0 0 0 0]
   3  [0 0 1 1 1 0 0 0]
   4  [0 0 0 1 1 1 0 0]
   5  [0 0 0 0 1 1 1 0]
   6  [0 0 0 0 0 1 1 1]
   7  [0 0 0 0 0 0 1 1]
```

In [16]:
def create_sliding_window_mask(seq_len, window_size):
    """
    Create a causal-free sliding window attention mask.

    Args:
        seq_len:     int — sequence length
        window_size: int — one-sided window (total window = 2 * window_size + 1)

    Returns:
        mask: (1, 1, seq_len, seq_len) — 1 for attend, 0 for block
    """
    # (seq_len, seq_len) — distance matrix
    positions = torch.arange(seq_len)
    distance = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()

    # 1 where distance ≤ window_size, 0 otherwise
    # (seq_len, seq_len) → (1, 1, seq_len, seq_len)
    mask = (distance <= window_size).float().unsqueeze(0).unsqueeze(0)
    return mask


class ModernBERTAttention(nn.Module):
    """
    Attention with RoPE, supporting both global and local (sliding window) modes.
    Reference: Warner et al., 2024.
    """

    def __init__(self, embed_dim, num_heads, max_len=8192, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = math.sqrt(self.head_dim)

        # QKV as a single fused projection (common in modern implementations)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, 3 * embed_dim)
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)

        # Output projection: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim, max_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None, window_size=None):
        # x: (batch_num, seq_len, embed_dim)
        # padding_mask: (batch_num, 1, 1, seq_len) — 1=valid, 0=pad
        # window_size: int or None — if set, use local sliding window attention
        batch_num, seq_len, _ = x.shape

        # Fused QKV projection: (batch_num, seq_len, 3 * embed_dim)
        qkv = self.qkv(x)
        # Split: 3 × (batch_num, seq_len, embed_dim)
        q, k, v = qkv.chunk(3, dim=-1)

        # Reshape for multi-head: (batch_num, num_heads, seq_len, head_dim)
        q = q.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to queries and keys
        cos, sin = self.rope(seq_len)
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        q, k = apply_rotary_pos_emb(q, k, cos.to(x.device), sin.to(x.device))

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, seq_len)
        # → scores: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale

        # Apply sliding window mask for local attention
        if window_size is not None:
            # (1, 1, seq_len, seq_len)
            window_mask = create_sliding_window_mask(seq_len, window_size).to(x.device)
            scores = scores.masked_fill(window_mask == 0, float("-inf"))

        # Apply padding mask
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask == 0, float("-inf"))

        # Softmax + dropout → attention weights
        # (batch_num, num_heads, seq_len, seq_len)
        attn = self.dropout(F.softmax(scores, dim=-1))

        # Weighted sum of values
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, embed_dim)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_num, seq_len, -1)

        # Output projection: (batch_num, seq_len, embed_dim)
        return self.out_proj(out)


# Demo: local vs global attention
_attn = ModernBERTAttention(embed_dim=256, num_heads=8, max_len=128)
_x = torch.randn(2, 32, 256)

_global_out = _attn(_x, window_size=None)
_local_out = _attn(_x, window_size=4)
print(f"Global attention output: {_global_out.shape}")  # (2, 32, 256)
print(f"Local attention output:  {_local_out.shape}")   # (2, 32, 256)

# Visualize sliding window mask
mask = create_sliding_window_mask(8, window_size=2)
print(f"\nSliding window mask (seq_len=8, window=2):")
print(mask[0, 0].int())

Global attention output: torch.Size([2, 32, 256])
Local attention output:  torch.Size([2, 32, 256])

Sliding window mask (seq_len=8, window=2):
tensor([[1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0],
        [0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1]], dtype=torch.int32)


## 3.4 Unpadding: Eliminating Wasted Computation

Standard batching **pads** shorter sequences to the batch maximum, wasting compute
on padding tokens. **Unpadding** concatenates only the real tokens and tracks their
boundaries.

**Example:**
```
Standard (padded):              Unpadded:
[CLS] The cat [PAD] [PAD]      [CLS] The cat [CLS] A big dog sat [CLS] Hi
[CLS] A big dog sat [PAD]      ← all real tokens concatenated
[CLS] Hi [PAD] [PAD] [PAD]

Total compute: 3×5 = 15 tokens  Total compute: 3+5+2 = 10 tokens (33% saved)
```

The unpadding approach:
1. **`unpad_input`**: Remove padding, flatten to 1D, record indices
2. **Process**: Run attention on the variable-length concatenated sequence
3. **`pad_input`**: Scatter results back to padded positions

**Reference:** Originally from Ivanov et al. (2021), widely adopted in FlashAttention.

In [17]:
def unpad_input(hidden_states, attention_mask):
    """
    Remove padding tokens and concatenate all real tokens into a flat sequence.

    Args:
        hidden_states:  (batch_num, seq_len, embed_dim)
        attention_mask: (batch_num, seq_len) — 1 for real, 0 for pad

    Returns:
        unpadded: (total_tokens, embed_dim) — only real tokens
        indices:  (total_tokens,) — original flat indices for scatter back
        cu_seqlens: (batch_num + 1,) — cumulative sequence lengths
        max_seqlen: int — max sequence length in batch
    """
    # (total_tokens,) — flat indices of non-padding positions
    indices = torch.nonzero(attention_mask.flatten(), as_tuple=False).flatten()

    # Flatten and gather: (batch_num * seq_len, embed_dim) → (total_tokens, embed_dim)
    batch_num, seq_len, embed_dim = hidden_states.shape
    flat = hidden_states.reshape(-1, embed_dim)
    unpadded = flat.index_select(0, indices)

    # Compute cumulative sequence lengths for each sample
    # seqlens: (batch_num,) — number of real tokens per sample
    seqlens = attention_mask.sum(dim=1).long()
    cu_seqlens = F.pad(torch.cumsum(seqlens, dim=0), (1, 0))
    max_seqlen = seqlens.max().item()

    return unpadded, indices, cu_seqlens, max_seqlen


def pad_input(unpadded, indices, batch_num, seq_len):
    """
    Scatter unpadded tokens back to their original padded positions.

    Args:
        unpadded: (total_tokens, embed_dim)
        indices:  (total_tokens,) — original flat indices
        batch_num: int
        seq_len: int

    Returns:
        padded: (batch_num, seq_len, embed_dim)
    """
    embed_dim = unpadded.shape[-1]
    # (batch_num * seq_len, embed_dim) — zero-initialized
    padded = torch.zeros(batch_num * seq_len, embed_dim, device=unpadded.device)
    # Scatter real tokens back
    padded.index_copy_(0, indices, unpadded)
    # Reshape: (batch_num, seq_len, embed_dim)
    return padded.view(batch_num, seq_len, embed_dim)


# Demo: unpad → process → pad round-trip
_x = torch.randn(3, 8, 256)
_mask = torch.tensor([
    [1, 1, 1, 0, 0, 0, 0, 0],  # 3 real tokens
    [1, 1, 1, 1, 1, 0, 0, 0],  # 5 real tokens
    [1, 1, 0, 0, 0, 0, 0, 0],  # 2 real tokens
])

unpadded, indices, cu_seqlens, max_seqlen = unpad_input(_x, _mask)
print(f"Original:       {_x.shape}")          # (3, 8, 256)
print(f"Unpadded:       {unpadded.shape}")     # (10, 256) — 3+5+2=10 real tokens
print(f"Cu_seqlens:     {cu_seqlens}")         # [0, 3, 8, 10]
print(f"Max seq length: {max_seqlen}")         # 5

repadded = pad_input(unpadded, indices, 3, 8)
print(f"Re-padded:      {repadded.shape}")     # (3, 8, 256)

# Verify round-trip: non-padding positions should match
real_mask = _mask.bool().unsqueeze(-1).expand_as(_x)
print(f"Round-trip match: {torch.allclose(_x[real_mask], repadded[real_mask])}")

Original:       torch.Size([3, 8, 256])
Unpadded:       torch.Size([10, 256])
Cu_seqlens:     tensor([ 0,  3,  8, 10])
Max seq length: 5
Re-padded:      torch.Size([3, 8, 256])
Round-trip match: True


## 3.5 Full ModernBERT Encoder

Assembling all modern components with **Pre-LayerNorm** architecture:

$$\text{output} = x + \text{SubLayer}(\text{LayerNorm}(x))$$

Pre-LN (Xiong et al., 2020) normalizes **before** the sub-layer, which:
- Stabilizes training (especially for deep models)
- Allows higher learning rates
- Removes the need for careful warmup

**Full architecture:**
```
Token Embedding (no positional — RoPE is applied inside attention)
     ↓
For each layer i:
  ├── Pre-LN → Attention (RoPE)
  │   └── Local (window) if i % 3 != 0, else Global
  ├── Residual connection
  ├── Pre-LN → GeGLU FFN
  └── Residual connection
     ↓
Final LayerNorm
     ↓
Output: (batch_num, seq_len, embed_dim)
```

**Note on Flash Attention:** In production, ModernBERT uses Flash Attention 2
(Dao, 2023) for hardware-efficient attention. We implement the mathematical equivalent
in standard PyTorch for clarity.

In [18]:
class ModernBERTEncoderLayer(nn.Module):
    """
    Single ModernBERT layer: Pre-LN + RoPE Attention + GeGLU FFN.
    Supports both local (sliding window) and global attention modes.
    """

    def __init__(self, embed_dim, num_heads, hidden_dim, max_len=8192, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = ModernBERTAttention(embed_dim, num_heads, max_len, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = GeGLU(embed_dim, hidden_dim, dropout)

    def forward(self, x, padding_mask=None, window_size=None):
        # x: (batch_num, seq_len, embed_dim)

        # Pre-LN → Attention → Residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        normed = self.norm1(x)
        x = x + self.attention(normed, padding_mask, window_size)

        # Pre-LN → GeGLU FFN → Residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        normed = self.norm2(x)
        x = x + self.ffn(normed)

        return x


class ModernBERT(nn.Module):
    """
    ModernBERT: modern bidirectional encoder with RoPE, GeGLU, alternating attention.
    Reference: Warner et al., 2024.
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=768,
        num_layers=22,
        num_heads=12,
        hidden_dim=2048,
        max_len=8192,
        local_window_size=128,
        global_every_n=3,
        dropout=0.0,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.local_window_size = local_window_size
        self.global_every_n = global_every_n

        # Token embedding only — RoPE handles position inside attention
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding_norm = nn.LayerNorm(embed_dim)

        # Stacked encoder layers
        self.layers = nn.ModuleList(
            [
                ModernBERTEncoderLayer(embed_dim, num_heads, hidden_dim, max_len, dropout)
                for _ in range(num_layers)
            ]
        )

        # Final layer norm (required for Pre-LN architecture)
        self.final_norm = nn.LayerNorm(embed_dim)

    def forward(self, token_ids, attention_mask=None):
        # token_ids:     (batch_num, seq_len)
        # attention_mask: (batch_num, seq_len) — 1=real, 0=pad

        batch_num, seq_len = token_ids.shape

        # Build padding mask: (batch_num, 1, 1, seq_len)
        if attention_mask is None:
            attention_mask = (token_ids != 0)
        padding_mask = attention_mask.unsqueeze(1).unsqueeze(2).float()

        # Token embedding: (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        x = self.embedding_norm(self.token_embedding(token_ids))

        # Pass through encoder layers with alternating local/global attention
        for i, layer in enumerate(self.layers):
            # Global attention every global_every_n-th layer, local otherwise
            if i % self.global_every_n == 0:
                window_size = None  # global
            else:
                window_size = self.local_window_size  # local sliding window

            # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
            x = layer(x, padding_mask, window_size)

        # Final normalization
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = self.final_norm(x)

        return x


# Build and test ModernBERT (small config for demo)
modern_bert = ModernBERT(
    vocab_size=VOCAB_SIZE,
    embed_dim=256,
    num_layers=6,
    num_heads=8,
    hidden_dim=512,
    max_len=512,
    local_window_size=16,
    global_every_n=3,
)

_tok = torch.randint(1, VOCAB_SIZE, (2, 64))
_mask = torch.ones(2, 64, dtype=torch.long)
_mask[0, 48:] = 0  # simulate padding
_mask[1, 56:] = 0

_out = modern_bert(_tok, _mask)
print(f"ModernBERT output: {_out.shape}")  # (2, 64, 256)
print(f"ModernBERT parameters: {sum(p.numel() for p in modern_bert.parameters()):,}")

# Show which layers use global vs local attention
for i in range(6):
    mode = "GLOBAL" if i % 3 == 0 else "LOCAL (window=16)"
    print(f"  Layer {i}: {mode}")

ModernBERT output: torch.Size([2, 64, 256])
ModernBERT parameters: 11,619,328
  Layer 0: GLOBAL
  Layer 1: LOCAL (window=16)
  Layer 2: LOCAL (window=16)
  Layer 3: GLOBAL
  Layer 4: LOCAL (window=16)
  Layer 5: LOCAL (window=16)


---
# 8) Architecture Comparison & Summary

## Side-by-Side Comparison

| Feature | BERT (2019) | DeBERTa (2021) | ModernBERT (2024) |
|---------|-------------|----------------|-------------------|
| **Position Encoding** | Absolute (sinusoidal/learned) | Relative (disentangled) + absolute in EMD | RoPE (rotation-based relative) |
| **Attention** | Full self-attention | Disentangled c2c + c2p + p2c | Alternating local/global with RoPE |
| **FFN** | GELU (2 matrices) | GELU (2 matrices) | GeGLU (3 matrices, gated) |
| **Normalization** | Post-LayerNorm | Post-LayerNorm | Pre-LayerNorm |
| **Pre-training** | MLM + NSP | MLM (v1), RTD (v3) | MLM only |
| **Max Context** | 512 | 512 | 8,192 |
| **Key Insight** | Bidirectional context | Disentangle content & position | Apply modern LLM improvements |

## Evolution of Ideas

```
BERT (2019)
  │
  ├── Problem: Absolute positions conflate with content
  ▼
DeBERTa (2021)
  │  ├── Fix: Disentangle position from content in attention
  │  └── Fix: Inject absolute position only where needed (EMD)
  │
  ├── Problem: Inefficient for long sequences; outdated components
  ▼
ModernBERT (2024)
     ├── Fix: RoPE for efficient relative positions
     ├── Fix: GeGLU for better FFN expressiveness
     ├── Fix: Alternating attention for long-context efficiency
     ├── Fix: Unpadding to eliminate wasted computation
     └── Fix: Pre-LN for training stability
```

## Key Takeaways for Practitioners

1. **DeBERTa** is still the strongest encoder for fine-tuning on short-sequence tasks
   (GLUE, SuperGLUE) due to its disentangled attention mechanism.

2. **ModernBERT** excels at long-context tasks and inference efficiency, making it
   the best choice for retrieval, code understanding, and document classification.

3. **BERT** remains an excellent educational baseline and is still competitive on
   many tasks when properly fine-tuned (RoBERTa improvements help too).

4. The trend is clear: **separate content from position** (DeBERTa, RoPE),
   **use gated activations** (GeGLU), and **think about efficiency** (unpadding,
   local attention).